# 🎙️ Zayra — własny głos (Piper) · darmowy trening na Colab

Ten notebook wytrenuje **Twój własny głos** dla asystentki Zayry — za darmo, na GPU Google Colab (T4), a potem wyeksportuje go do plików `.onnx` + `.onnx.json`, które wgrywasz bezpośrednio do Zayry. Po treningu głos działa **lokalnie w przeglądarce na słabym CPU**, offline, bez żadnych płatnych API.

**Zanim zaczniesz:**
1. Nagraj **10–30 minut** czystego głosu (mono, WAV, 22050 Hz, 16-bit). Podziel na pliki po **3–12 s**, nazwij `1.wav`, `2.wav`, … i spakuj do **`wavs.zip`**.
2. Wgraj `wavs.zip` na swój **Google Drive** (lub wgrasz go bezpośrednio w kroku 4).
3. Ustaw **Runtime → Change runtime type → T4 GPU**.
4. Uruchamiaj komórki **po kolei** (Shift+Enter).

> ⚠️ Klonuj wyłącznie głosy **za zgodą** — swój własny albo osoby, która wyraziła zgodę.

Pipeline oparty na sprawdzonym notebooku `rmcpantoja/piper` (fine-tune Piper + Whisper + eksport ONNX).

In [ ]:
#@title 1. Sprawdź GPU + anti-disconnect
!nvidia-smi
import IPython
display(IPython.display.Javascript('''
function ClickConnect(){document.querySelector("colab-toolbar-button#connect")?.click()}
setInterval(ClickConnect,60000)
'''))

In [ ]:
#@title 2. Zamontuj Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
#@title 3. Zainstaluj Piper + zależności (chwila potrwa)
!git clone -q https://github.com/rmcpantoja/piper
%cd /content/piper/src/python
!wget -q "https://raw.githubusercontent.com/coqui-ai/TTS/dev/TTS/bin/resample.py"
!pip install -q cython>=0.29.0 piper-phonemize==1.1.0 librosa>=0.9.2 numpy==1.26 onnxruntime>=1.15.0 pytorch-lightning faster-whisper
!pip install -q --upgrade gdown
!bash build_monotonic_align.sh
print("Gotowe!")

In [ ]:
#@title 4. Ustawienia głosu + pobierz polski model bazowy
import json, os
model_name = "zayra" #@param {type:"string"}
base_voice = "gosia-medium" #@param ["gosia-medium", "darkman-medium", "mc_speech-medium"]
quality = "medium" #@param ["medium", "high", "x-low"]
sample_rate = 22050 #@param [22050, 16000]
batch_size = 12 #@param {type:"integer"}
max_epochs = 10000 #@param {type:"integer"}

output_dir = f"/content/drive/MyDrive/zayra-piper/{model_name}"
os.makedirs(output_dir, exist_ok=True)
os.makedirs("/content/audio_cache", exist_ok=True)

pm = json.load(open('/content/piper/notebooks/pretrained_models.json'))
url = pm['pl'][base_voice + " (fine-tuned)"]
print("Pobieram model bazowy:", base_voice)
!wget -q "{url}" -O /content/pretrained.ckpt
assert os.path.exists('/content/pretrained.ckpt'), "Nie pobrano modelu bazowego!"
print("Model bazowy gotowy.")

In [ ]:
#@title 5. Wgraj dataset (wavs.zip z Drive lub bezpośrednio)
import os, zipfile, io
zip_path = "/content/drive/MyDrive/wavs.zip" #@param {type:"string"}
os.makedirs('/content/dataset/wavs', exist_ok=True)
if os.path.exists(zip_path):
    !unzip -q -j "{zip_path}" -d /content/dataset/wavs
    print("Rozpakowano z Drive.")
else:
    from google.colab import files
    print("Wybierz plik wavs.zip…")
    up = files.upload()
    for name, data in up.items():
        if name.endswith('.zip'):
            zipfile.ZipFile(io.BytesIO(data)).extractall('/content/dataset/wavs')
    print("Rozpakowano uploaded zip.")
wavs = [f for f in os.listdir('/content/dataset/wavs') if f.endswith('.wav')]
print(f"Liczba plików WAV: {len(wavs)}")
assert len(wavs) > 0, "Brak plików WAV!" 

In [ ]:
#@title 6. Transkrypcja (Whisper, polski) — albo wgraj własny metadata.csv
import os
use_whisper = True #@param {type:"boolean"}
if use_whisper:
    import torch
    from faster_whisper import WhisperModel
    from tqdm import tqdm
    device = "cuda" if torch.cuda.is_available() else "cpu"
    whisper = WhisperModel("large-v3", device=device, compute_type="float16")
    files = sorted(f for f in os.listdir('/content/dataset/wavs') if f.endswith('.wav'))
    with open('/content/dataset/metadata.csv', 'w', encoding='utf-8') as meta:
        for af in tqdm(files):
            segments, _ = whisper.transcribe(os.path.join('/content/dataset/wavs', af), language='pl')
            text = ' '.join(s.text for s in segments).strip().replace('\n', ' ')
            meta.write(f"{af}|{text}\n")
    del whisper
    print("Transkrypcja gotowa. Możesz podejrzeć /content/dataset/metadata.csv i poprawić tekst ręcznie.")
else:
    print("Używam własnego metadata.csv (wgraj go do /content/dataset/metadata.csv).")

In [ ]:
#@title 7. Preprocess datasetu
%cd /content/piper/src/python
resample = False #@param {type:"boolean"}
if resample:
    !python resample.py --input_dir "/content/dataset/wavs" --output_dir "/content/dataset/wavs_rs" --output_sr {sample_rate} --file_ext "wav"
    !mv /content/dataset/wavs_rs/* /content/dataset/wavs
!python -m piper_train.preprocess \
  --language pl \
  --input-dir /content/dataset \
  --cache-dir "/content/audio_cache" \
  --output-dir "{output_dir}" \
  --dataset-name "{model_name}" \
  --dataset-format ljspeech \
  --sample-rate {sample_rate} \
  --single-speaker
print("Preprocess gotowy!")

In [ ]:
#@title 8. Trening (fine-tune). Zatrzymaj, gdy próbki brzmią dobrze.
%cd /content/piper/src/python
%load_ext tensorboard
%tensorboard --logdir "{output_dir}"
!python -m piper_train \
  --dataset-dir "{output_dir}" \
  --accelerator gpu --devices 1 \
  --batch-size {batch_size} \
  --validation-split 0.01 --num-test-examples 1 \
  --quality {quality} \
  --checkpoint-epochs 5 \
  --num_ckpt 0 --save_last True \
  --log_every_n_steps 200 \
  --max_epochs {max_epochs} \
  --resume_from_checkpoint /content/pretrained.ckpt \
  --precision 32

In [ ]:
#@title 9. Eksport do ONNX (+ config)
%cd /content/piper/src/python
import glob, os, re, shutil
ckpts = glob.glob(f"{output_dir}/lightning_logs/**/checkpoints/last.ckpt", recursive=True)
assert ckpts, "Brak checkpointu — najpierw wytrenuj model."
ckpt = sorted(ckpts, key=lambda x: int(re.findall(r'version_(\d+)', x)[-1]))[-1]
out_onnx = f"/content/{model_name}.onnx"
!python -m piper_train.export_onnx "{ckpt}" "{out_onnx}"
cfg = out_onnx + ".json"
if not os.path.exists(cfg):
    shutil.copy(f"{output_dir}/config.json", cfg)
print("ONNX:", out_onnx)
print("Config:", cfg)

In [ ]:
#@title 10. Pobierz pliki głosu (i/lub zapisz na Drive)
from google.colab import files
import shutil, os
save_to_drive = True #@param {type:"boolean"}
files.download(out_onnx)
files.download(out_onnx + ".json")
if save_to_drive:
    dst = "/content/drive/MyDrive/zayra-piper"
    os.makedirs(dst, exist_ok=True)
    shutil.copy(out_onnx, dst); shutil.copy(out_onnx + ".json", dst)
    print("Zapisano też na Drive:", dst)

## ✅ Gotowe — wgraj głos do Zayry

1. Otwórz Zayrę → ikonka **ustawień głosu** (suwaki) w nagłówku.
2. Sekcja **Własny głos** → wgraj **`zayra.onnx`** i **`zayra.onnx.json`** → **Zainstaluj**.
3. Kliknij **Posłuchaj**, żeby przetestować. Od teraz Zayra mówi Twoim głosem — lokalnie, offline.

**Wskazówki jakości:**
- Więcej czystych danych (20–30 min) = lepszy głos.
- Słuchaj próbek w TensorBoard podczas treningu; zatrzymaj, gdy brzmi naturalnie (zwykle kilkaset–1000 epok).
- Jeśli głos jest niewyraźny, popraw transkrypcję w `metadata.csv` i dotrenuj (opcja *Continue training* w oryginalnym notebooku).
- Model bazowy `gosia` (żeński) / `darkman` (męski) wybierz zbliżony do barwy docelowej.